In [1]:

!nvidia-smi
import sys
print(f"Python version: {sys.version}")


Wed Sep 23 11:49:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:

!pip install -q ultralytics opencv-python-headless pandas numpy pyyaml

import torch, cv2, ultralytics
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
print(f"OpenCV version: {cv2.__version__}")
print(f"Ultralytics version: {ultralytics.__version__}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.7/46.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 9.8 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU name: Tesla T4
OpenCV version: 5.0.0
Ultralytics version: 8.4.160


In [3]:

from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [10]:
import os


PROJECT_ROOT = "/content/penalty-system"
DRIVE_BACKUP_ROOT = "/content/drive/MyDrive/penalty-system-backup"
VIDEO_PATH = "/content/drive/MyDrive/traffic.mp4"
MODEL_PATH = f"{PROJECT_ROOT}/models/license_plate_detector.pt"


ROBOFLOW_API_KEY = "YOUR_API_KEY_HERE"
ROBOFLOW_WORKSPACE = "ilpd"
ROBOFLOW_PROJECT = "indian-licence-plate-detection"
ROBOFLOW_VERSION = 4


DOWNLOAD_DATASET = True
RUN_AUTO_LABELING = True
RUN_TRAINING = True
FORCE_RETRAIN = True
RUN_INFERENCE = True
RUN_OCR = False


IMAGE_SIZE = 960
EPOCHS = 80
BATCH_SIZE = 8
PATIENCE = 20
PRETRAINED_WEIGHTS = "yolo11s.pt"
RUN_NAME = "plate_detector_v2_realworld"


CONFIDENCE_THRESHOLD = 0.40
ANNOTATED_VIDEO_PATH = f"{PROJECT_ROOT}/outputs/annotated/traffic_with_plates.mp4"


REFERENCE_MODEL_ID = "keremberke/yolov5s-license-plate"

print("Configuration loaded.")


Configuration loaded.


In [5]:
FOLDERS = [
    f"{PROJECT_ROOT}/data/input",
    f"{PROJECT_ROOT}/data/images/train", f"{PROJECT_ROOT}/data/images/val", f"{PROJECT_ROOT}/data/images/test",
    f"{PROJECT_ROOT}/data/labels/train", f"{PROJECT_ROOT}/data/labels/val", f"{PROJECT_ROOT}/data/labels/test",
    f"{PROJECT_ROOT}/data/extracted_frames", f"{PROJECT_ROOT}/data/auto_labeled/images", f"{PROJECT_ROOT}/data/auto_labeled/labels",
    f"{PROJECT_ROOT}/models",
    f"{PROJECT_ROOT}/outputs/annotated", f"{PROJECT_ROOT}/outputs/plates",
    f"{PROJECT_ROOT}/outputs/frames", f"{PROJECT_ROOT}/outputs/metadata",
    f"{PROJECT_ROOT}/src", f"{PROJECT_ROOT}/notebooks",
]
for f in FOLDERS:
    os.makedirs(f, exist_ok=True)
os.makedirs(DRIVE_BACKUP_ROOT, exist_ok=True)
print("Project structure ready at:", PROJECT_ROOT)


Project structure ready at: /content/penalty-system


In [6]:

import shutil

if os.path.exists(DRIVE_BACKUP_ROOT) and os.listdir(DRIVE_BACKUP_ROOT):
    print("Found existing Drive backup — restoring...")
    for item in ["data.yaml"]:
        src = f"{DRIVE_BACKUP_ROOT}/{item}"
        if os.path.exists(src):
            shutil.copy(src, f"{PROJECT_ROOT}/{item}")
    model_backup = f"{DRIVE_BACKUP_ROOT}/license_plate_detector.pt"
    if os.path.exists(model_backup):
        shutil.copy(model_backup, MODEL_PATH)
        print("Restored trained model from backup.")
    data_backup = f"{DRIVE_BACKUP_ROOT}/data"
    if os.path.exists(data_backup):
        shutil.copytree(data_backup, f"{PROJECT_ROOT}/data", dirs_exist_ok=True)
        print("Restored dataset splits from backup.")
    if os.path.exists(MODEL_PATH) and not FORCE_RETRAIN:
        RUN_TRAINING = False
        DOWNLOAD_DATASET = False
        print("MODEL_PATH already restored -> skipping dataset download and training this run.")
        print("(set FORCE_RETRAIN = True in the config cell to override)")
else:
    print("No existing Drive backup found — this looks like a first run.")


Found existing Drive backup — restoring...
Restored trained model from backup.
Restored dataset splits from backup.
MODEL_PATH already restored -> skipping dataset download and training this run.
(set FORCE_RETRAIN = True in the config cell to override)


In [7]:
import cv2
import pandas as pd
from collections import Counter

def validate_yolo_dataset(images_dir, labels_dir, expected_classes={0: "license_plate"}):
    """Validates a YOLO-format dataset. Does NOT modify any files."""
    report = {
        "total_images": 0, "total_labels": 0, "missing_labels": [], "empty_labels": [],
        "invalid_bboxes": [], "invalid_class_ids": [], "image_dimension_issues": [],
        "class_distribution": Counter(), "image_dimensions": Counter(),
    }
    if not os.path.exists(images_dir):
        print(f"ERROR: images directory not found: {images_dir}")
        return report

    image_files = [f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    report["total_images"] = len(image_files)

    for img_file in image_files:
        img_path = os.path.join(images_dir, img_file)
        label_file = os.path.splitext(img_file)[0] + ".txt"
        label_path = os.path.join(labels_dir, label_file)

        img = cv2.imread(img_path)
        if img is None:
            report["image_dimension_issues"].append(img_file)
            continue
        h, w = img.shape[:2]
        report["image_dimensions"][(w, h)] += 1

        if not os.path.exists(label_path):
            report["missing_labels"].append(img_file)
            continue

        with open(label_path, "r") as f:
            lines = [l.strip() for l in f.readlines() if l.strip()]

        if len(lines) == 0:
            report["empty_labels"].append(img_file)
            continue

        report["total_labels"] += 1
        for line_num, line in enumerate(lines):
            parts = line.split()
            if len(parts) != 5:
                report["invalid_bboxes"].append((img_file, line_num, "wrong field count"))
                continue
            try:
                class_id = int(parts[0])
                x_c, y_c, bw, bh = map(float, parts[1:])
            except ValueError:
                report["invalid_bboxes"].append((img_file, line_num, "non-numeric value"))
                continue
            if class_id not in expected_classes:
                report["invalid_class_ids"].append((img_file, line_num, class_id))
            if not (0 <= x_c <= 1 and 0 <= y_c <= 1 and 0 < bw <= 1 and 0 < bh <= 1):
                report["invalid_bboxes"].append((img_file, line_num, "out-of-range coordinates"))
                continue
            report["class_distribution"][class_id] += 1
    return report


def print_validation_report(report, expected_classes={0: "license_plate"}):
    print(f"Total images: {report['total_images']}  |  With labels: {report['total_labels']}  |  "
          f"Missing labels: {len(report['missing_labels'])}  |  Invalid bboxes: {len(report['invalid_bboxes'])}  |  "
          f"Invalid class IDs: {len(report['invalid_class_ids'])}  |  Corrupt images: {len(report['image_dimension_issues'])}")
    print("Class distribution:")
    for cid, count in report["class_distribution"].items():
        name = expected_classes.get(cid, f"UNKNOWN({cid})")
        print(f"  {cid} ({name}): {count}")

print("Validation utilities defined.")


Validation utilities defined.


In [11]:
if DOWNLOAD_DATASET:
    !pip install -q roboflow
    from roboflow import Roboflow

    rf = Roboflow(api_key="yQZDI1bpMBtcdJSdZnUY")
    project = rf.workspace("ilpd").project("indian-licence-plate-detection")
    version = project.version(4)
    dataset = version.download("yolov11", location=f"{PROJECT_ROOT}/data_downloaded")
    print("Downloaded to:", dataset.location)
else:
    print("Skipping dataset download (DOWNLOAD_DATASET = False).")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.4/322.4 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 105.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.4/58.4 kB 6.3 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/penalty-system/data_downloaded in yolov11:: 100%|██████████| 1404/1404 [00:00<00:00, 8769.35it/s]

Downloaded to: /content/penalty-system/data_downloaded


In [12]:
if DOWNLOAD_DATASET:
    import shutil

    def merge_split(split, target_split):
        src_images = os.path.join(f"{PROJECT_ROOT}/data_downloaded", split, "images")
        src_labels = os.path.join(f"{PROJECT_ROOT}/data_downloaded", split, "labels")
        if not os.path.exists(src_images):
            return
        dst_images = f"{PROJECT_ROOT}/data/images/{target_split}"
        dst_labels = f"{PROJECT_ROOT}/data/labels/{target_split}"
        for fname in os.listdir(src_images):
            shutil.copy(os.path.join(src_images, fname), dst_images)
        for fname in os.listdir(src_labels):
            shutil.copy(os.path.join(src_labels, fname), dst_labels)

    for split in ["train", "valid", "test"]:
        merge_split(split, "val" if split == "valid" else split)

    def clean_labels_plate_only(labels_dir, output_labels_dir, target_class=0):
        os.makedirs(output_labels_dir, exist_ok=True)
        for label_file in os.listdir(labels_dir):
            if not label_file.endswith(".txt"):
                continue
            with open(os.path.join(labels_dir, label_file)) as f:
                lines = [l.strip() for l in f if l.strip()]
            kept = [l for l in lines if len(l.split()) == 5 and l.split()[0] == str(target_class)]
            with open(os.path.join(output_labels_dir, label_file), "w") as f:
                f.write("\n".join(kept))

    def build_final_split(images_dir, cleaned_labels_dir, out_images_dir, out_labels_dir):
        os.makedirs(out_images_dir, exist_ok=True)
        os.makedirs(out_labels_dir, exist_ok=True)
        kept = 0
        for label_file in os.listdir(cleaned_labels_dir):
            with open(os.path.join(cleaned_labels_dir, label_file)) as f:
                if not f.read().strip():
                    continue
            img_name = os.path.splitext(label_file)[0] + ".jpg"
            src_img = os.path.join(images_dir, img_name)
            if os.path.exists(src_img):
                shutil.copy(src_img, out_images_dir)
                shutil.copy(os.path.join(cleaned_labels_dir, label_file), out_labels_dir)
                kept += 1
        return kept

    # NOTE: "images"/"labels" stay as literal path segments (only the split suffix changes)
    # so Ultralytics' automatic images->labels path substitution works correctly.
    for split in ["train", "val", "test"]:
        clean_labels_plate_only(f"{PROJECT_ROOT}/data/labels/{split}", f"{PROJECT_ROOT}/data/labels_cleaned/{split}")
        n = build_final_split(
            f"{PROJECT_ROOT}/data/images/{split}", f"{PROJECT_ROOT}/data/labels_cleaned/{split}",
            f"{PROJECT_ROOT}/data/images/{split}_final", f"{PROJECT_ROOT}/data/labels/{split}_final"
        )
        print(f"[{split}] final images kept: {n}")
else:
    print("Skipping dataset cleaning (DOWNLOAD_DATASET = False).")


[train] final images kept: 336
[val] final images kept: 13
[test] final images kept: 8


In [13]:
if not os.path.exists(VIDEO_PATH):
    print(f"ERROR: video not found at {VIDEO_PATH} — check VIDEO_PATH in the config cell.")
else:
    cap = cv2.VideoCapture(VIDEO_PATH)
    if not cap.isOpened():
        print("ERROR: OpenCV could not open this video file — check codec/format.")
    else:
        fps = cap.get(cv2.CAP_PROP_FPS)
        frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        duration = frame_count / fps if fps > 0 else 0
        print(f"FPS: {fps:.2f}, Frames: {int(frame_count)}, Resolution: {width}x{height}, Duration: {duration:.1f}s")
    cap.release()


FPS: 30.00, Frames: 613, Resolution: 3840x2160, Duration: 20.4s


In [16]:
if RUN_AUTO_LABELING and os.path.exists(VIDEO_PATH):
    !pip install -q yolov5
    import yolov5


    import torch as _torch
    _original_torch_load = _torch.load
    _torch.load = lambda *args, **kwargs: _original_torch_load(*args, **{**kwargs, "weights_only": False})

    reference_model = yolov5.load(REFERENCE_MODEL_ID)
    reference_model.conf = 0.25
    reference_model.iou = 0.45

    _torch.load = _original_torch_load

    EXTRACT_DIR = f"{PROJECT_ROOT}/data/extracted_frames"
    cap = cv2.VideoCapture(VIDEO_PATH)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    NUM_FRAMES_TO_EXTRACT = 40
    frame_indices = [int(total_frames * i / NUM_FRAMES_TO_EXTRACT) for i in range(NUM_FRAMES_TO_EXTRACT)]

    extracted = 0
    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            cv2.imwrite(f"{EXTRACT_DIR}/traffic_frame_{idx:05d}.jpg", frame)
            extracted += 1
    cap.release()
    print(f"Extracted {extracted} frames from traffic video.")

    AUTO_LABEL_DIR = f"{PROJECT_ROOT}/data/auto_labeled"
    auto_labeled_count = 0
    for fname in os.listdir(EXTRACT_DIR):
        img_path = os.path.join(EXTRACT_DIR, fname)
        frame = cv2.imread(img_path)
        h, w = frame.shape[:2]

        results = reference_model(frame, size=640)
        preds = results.pred[0]
        if len(preds) == 0:
            continue

        lines = []
        for det in preds:
            x1, y1, x2, y2, conf, cls = det.tolist()
            if conf < 0.35:
                continue
            x_center = ((x1 + x2) / 2) / w
            y_center = ((y1 + y2) / 2) / h
            bw = (x2 - x1) / w
            bh = (y2 - y1) / h
            lines.append(f"0 {x_center:.6f} {y_center:.6f} {bw:.6f} {bh:.6f}")

        if lines:
            cv2.imwrite(f"{AUTO_LABEL_DIR}/images/{fname}", frame)
            label_name = os.path.splitext(fname)[0] + ".txt"
            with open(f"{AUTO_LABEL_DIR}/labels/{label_name}", "w") as f:
                f.write("\n".join(lines))
            auto_labeled_count += 1

    print(f"Auto-labeled {auto_labeled_count}/{extracted} real traffic frames.")

    import shutil
    for fname in os.listdir(f"{AUTO_LABEL_DIR}/images"):
        shutil.copy(f"{AUTO_LABEL_DIR}/images/{fname}", f"{PROJECT_ROOT}/data/images/train_final")
        label_name = os.path.splitext(fname)[0] + ".txt"
        shutil.copy(f"{AUTO_LABEL_DIR}/labels/{label_name}", f"{PROJECT_ROOT}/data/labels/train_final")
    print("Merged auto-labeled frames into train_final.")
    print("New train count:", len(os.listdir(f"{PROJECT_ROOT}/data/images/train_final")))
else:
    print("Skipping auto-labeling (RUN_AUTO_LABELING = False or video not found).")

Extracted 40 frames from traffic video.


`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
`torch.cuda.amp.autocast(arg

Auto-labeled 40/40 real traffic frames.
Merged auto-labeled frames into train_final.
New train count: 376


`torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.


In [17]:
import yaml

data_yaml_content = {
    "path": f"{PROJECT_ROOT}/data",
    "train": "images/train_final",
    "val": "images/val_final",
    "test": "images/test_final",
    "names": {0: "license_plate"},
}
with open(f"{PROJECT_ROOT}/data.yaml", "w") as f:
    yaml.dump(data_yaml_content, f, default_flow_style=False, sort_keys=False)

print("data.yaml written:")
with open(f"{PROJECT_ROOT}/data.yaml") as f:
    print(f.read())


data.yaml written:
path: /content/penalty-system/data
train: images/train_final
val: images/val_final
test: images/test_final
names:
  0: license_plate



In [18]:
for split in ["train", "val", "test"]:
    images_dir = f"{PROJECT_ROOT}/data/images/{split}_final"
    labels_dir = f"{PROJECT_ROOT}/data/labels/{split}_final"
    if os.path.exists(images_dir) and len(os.listdir(images_dir)) > 0:
        print(f"\n===== {split.upper()} =====")
        report = validate_yolo_dataset(images_dir, labels_dir)
        print_validation_report(report)
    else:
        print(f"\n[{split}] EMPTY — no usable images.")



===== TRAIN =====
Total images: 376  |  With labels: 376  |  Missing labels: 0  |  Invalid bboxes: 0  |  Invalid class IDs: 0  |  Corrupt images: 0
Class distribution:
  0 (license_plate): 513

===== VAL =====
Total images: 13  |  With labels: 13  |  Missing labels: 0  |  Invalid bboxes: 0  |  Invalid class IDs: 0  |  Corrupt images: 0
Class distribution:
  0 (license_plate): 21

===== TEST =====
Total images: 8  |  With labels: 8  |  Missing labels: 0  |  Invalid bboxes: 0  |  Invalid class IDs: 0  |  Corrupt images: 0
Class distribution:
  0 (license_plate): 16


In [20]:
if RUN_TRAINING:
    from ultralytics import YOLO

    device = 0 if torch.cuda.is_available() else "cpu"
    print(f"Training on: {'GPU - ' + torch.cuda.get_device_name(0) if device == 0 else 'CPU (slow)'}")

    model = YOLO(PRETRAINED_WEIGHTS)
    results = model.train(
        data=f"{PROJECT_ROOT}/data.yaml",
        imgsz=IMAGE_SIZE,
        epochs=EPOCHS,
        batch=BATCH_SIZE,
        patience=PATIENCE,
        device=device,
        project=f"{PROJECT_ROOT}/outputs/training_runs",
        name=RUN_NAME,
        exist_ok=True,
        verbose=True,
    )

    metrics = model.val(data=f"{PROJECT_ROOT}/data.yaml", imgsz=IMAGE_SIZE, device=device)
    print("\n=== VALIDATION METRICS ===")
    print(f"Precision: {metrics.box.mp:.4f}  Recall: {metrics.box.mr:.4f}  "
          f"mAP50: {metrics.box.map50:.4f}  mAP50-95: {metrics.box.map:.4f}")

    import shutil
    best_weights_path = f"{PROJECT_ROOT}/outputs/training_runs/{RUN_NAME}/weights/best.pt"
    shutil.copy(best_weights_path, MODEL_PATH)

    check_model = YOLO(MODEL_PATH)
    assert check_model.names == {0: "license_plate"}, "Unexpected class mapping!"
    print(f"\nBest weights saved to: {MODEL_PATH}")
    print("Classes:", check_model.names)
else:
    print("Skipping training (RUN_TRAINING = False).")
    if os.path.exists(MODEL_PATH):
        print(f"Using existing model at: {MODEL_PATH}")
    else:
        print("WARNING: no trained model found — RUN_INFERENCE will fail until one exists.")


Training on: GPU - Tesla T4
Ultralytics 8.4.160 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/penalty-system/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_detector_v

In [21]:
import shutil

if os.path.exists(MODEL_PATH):
    shutil.copy(MODEL_PATH, f"{DRIVE_BACKUP_ROOT}/license_plate_detector.pt")
if os.path.exists(f"{PROJECT_ROOT}/data.yaml"):
    shutil.copy(f"{PROJECT_ROOT}/data.yaml", f"{DRIVE_BACKUP_ROOT}/data.yaml")

for split in ["train", "val", "test"]:
    for kind in ["images", "labels"]:
        src_dir = f"{PROJECT_ROOT}/data/{kind}/{split}_final"
        dst_dir = f"{DRIVE_BACKUP_ROOT}/data/{kind}/{split}_final"
        if os.path.exists(src_dir):
            shutil.copytree(src_dir, dst_dir, dirs_exist_ok=True)

print("Backup to Drive complete:", DRIVE_BACKUP_ROOT)


Backup to Drive complete: /content/drive/MyDrive/penalty-system-backup


In [22]:
import time

def run_plate_detection(video_path, model_path, confidence_threshold, annotated_output_path):
    from ultralytics import YOLO
    if not os.path.exists(video_path):
        print(f"ERROR: video not found at {video_path}")
        return []
    if not os.path.exists(model_path):
        print(f"ERROR: model not found at {model_path}")
        return []

    model = YOLO(model_path)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print("ERROR: could not open video.")
        return []

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    os.makedirs(os.path.dirname(annotated_output_path), exist_ok=True)
    writer = cv2.VideoWriter(annotated_output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))

    all_detections = []
    frame_number = 0
    start_time = time.time()

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        try:
            results = model.predict(frame, conf=confidence_threshold, verbose=False)[0]
        except Exception as e:
            print(f"WARNING: inference failed on frame {frame_number}: {e}")
            frame_number += 1
            continue

        timestamp_seconds = frame_number / fps
        if results.boxes is not None and len(results.boxes) > 0:
            for box in results.boxes:
                x1, y1, x2, y2 = box.xyxy[0].tolist()
                conf = float(box.conf[0])
                if x2 <= x1 or y2 <= y1:
                    continue
                all_detections.append({
                    "frame_number": frame_number, "timestamp_seconds": round(timestamp_seconds, 2),
                    "plate_x1": round(x1, 1), "plate_y1": round(y1, 1),
                    "plate_x2": round(x2, 1), "plate_y2": round(y2, 1),
                    "plate_confidence": round(conf, 3),
                })
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (0, 255, 0), 2)
                cv2.putText(frame, f"License Plate {conf:.2f}", (int(x1), max(int(y1) - 8, 0)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        writer.write(frame)
        frame_number += 1
        if frame_number % 50 == 0:
            print(f"  Processed {frame_number}/{total_frames} frames...")

    cap.release()
    writer.release()
    elapsed = time.time() - start_time
    print(f"\nDone. {frame_number} frames in {elapsed:.1f}s ({frame_number/max(elapsed,0.01):.1f} FPS).")
    print(f"Total detections above {confidence_threshold}: {len(all_detections)}")
    return all_detections


if RUN_INFERENCE:
    detections = run_plate_detection(VIDEO_PATH, MODEL_PATH, CONFIDENCE_THRESHOLD, ANNOTATED_VIDEO_PATH)
else:
    detections = []
    print("Skipping inference (RUN_INFERENCE = False).")


  Processed 50/613 frames...
  Processed 100/613 frames...
  Processed 150/613 frames...
  Processed 200/613 frames...
  Processed 250/613 frames...
  Processed 300/613 frames...
  Processed 350/613 frames...
  Processed 400/613 frames...
  Processed 450/613 frames...
  Processed 500/613 frames...
  Processed 550/613 frames...
  Processed 600/613 frames...

Done. 613 frames in 101.5s (6.0 FPS).
Total detections above 0.4: 1418


In [24]:
# Sanity-check results
if detections:
    df = pd.DataFrame(detections)
    print(f"Frames with at least one detection: {df['frame_number'].nunique()}")
    print(f"Average confidence: {df['plate_confidence'].mean():.3f}")
    print(f"Confidence range: {df['plate_confidence'].min():.3f} - {df['plate_confidence'].max():.3f}")

    metadata_csv_path = f"{PROJECT_ROOT}/outputs/metadata/detections.csv"
    df.to_csv(metadata_csv_path, index=False)
    print(f"Saved metadata: {metadata_csv_path}")
else:
    print("No detections recorded — either the confidence threshold filtered everything out,")
    print("or the model isn't finding plates in this footage. Not necessarily a code error.")

if os.path.exists(ANNOTATED_VIDEO_PATH):
    size_mb = os.path.getsize(ANNOTATED_VIDEO_PATH) / (1024 * 1024)
    print(f"Annotated video: {ANNOTATED_VIDEO_PATH} ({size_mb:.1f} MB)")

Frames with at least one detection: 613
Average confidence: 0.694
Confidence range: 0.401 - 0.837
Saved metadata: /content/penalty-system/outputs/metadata/detections.csv
Annotated video: /content/penalty-system/outputs/annotated/traffic_with_plates.mp4 (211.6 MB)


In [25]:
# Visual spot-check
import matplotlib.pyplot as plt

if os.path.exists(ANNOTATED_VIDEO_PATH):
    cap = cv2.VideoCapture(ANNOTATED_VIDEO_PATH)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_indices = [int(total * f) for f in [0.1, 0.3, 0.5, 0.7, 0.9]] if total > 0 else []

    fig, axes = plt.subplots(1, len(sample_indices), figsize=(20, 4))
    for ax, idx in zip(axes, sample_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"Frame {idx}")
            ax.axis("off")
    cap.release()
    plt.tight_layout()
    plt.show()


In [26]:

from google.colab import files

try:
    files.download(ANNOTATED_VIDEO_PATH)
except Exception as e:
    print("Direct download failed/slow, copying to Drive instead:", e)
    import shutil
    shutil.copy(ANNOTATED_VIDEO_PATH, "/content/drive/MyDrive/traffic_with_plates.mp4")
    print("Saved to Drive — download from drive.google.com")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>